# 7. Test 80/20 models on external validation datasets

This notebook evaluates saved XGB and ENet 80/20 models on external validation datasets.

The evaluation is split into **four independent, highlighted evaluation cells** so the slow work is checkpointed after each model folder finishes:

1. `../results_xgb`
2. `../results_enet`
3. `../res_enet_zbig`
4. `../res_xgb_zbig`

Each evaluation cell saves its own prediction and metric files immediately, then refreshes the combined files in `../results_comparison_all_models/`. This makes the notebook safer to run in pieces: once a section finishes, its results are already on disk.

Main combined outputs for notebook 8 are saved to `../results_comparison_all_models/`:

- `eval_predictions_long.parquet` / `.csv` fallback
- `eval_metrics_by_protein.csv`
- `eval_metrics_by_tissue.csv`
- `model_inventory.csv`
- `feature_coverage.csv`

Per-folder outputs are saved with names like:

- `results_xgb__eval_predictions_long.parquet` / `.csv` fallback
- `results_xgb__eval_metrics_by_protein.csv`
- `results_xgb__feature_coverage.csv`

The follow-up notebook `8.prot_model_val.ipynb` reads the combined saved predictions/metrics and makes the final partition plots.


In [2]:
import os
import re
import sys
import json
import glob
import lzma
import pickle
import warnings
from pathlib import Path

import h5py
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from scipy.spatial.distance import cosine
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Fix for loading numpy 2.0 pickles in numpy 1.x environments.
try:
    import numpy.core.numeric
    import numpy.core.multiarray
    sys.modules["numpy._core"] = sys.modules.get("numpy.core")
    sys.modules["numpy._core.numeric"] = numpy.core.numeric
    sys.modules["numpy._core.multiarray"] = numpy.core.multiarray
except Exception:
    pass

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


In [3]:
# ============================================================
# Configuration
# ============================================================

VAL_DATA_DIR = Path("../val_data")
OUTPUT_DIR = Path("../results_comparison_all_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keys are intentionally the folder-style names requested for split evaluation.
MODEL_DIRS = {
    "results_xgb": Path("../results_xgb"),
    "results_enet": Path("../results_enet"),
    "res_enet_zbig": Path("../res_enet_zbig"),
    "res_xgb_zbig": Path("../res_xgb_zbig"),
}

EVALUATION_ORDER = [
    "results_xgb",
    "results_enet",
    "res_enet_zbig",
    "res_xgb_zbig",
]

# Expected validation dataset prefixes. The notebook will skip missing prefixes gracefully.
DATASET_PREFIXES = ["LabelFree", "iTRAQ", "TMT"]

# Set to an integer for a quick smoke test; set to None for full evaluation.
MAX_MODELS_TOTAL = None
MAX_MODELS_PER_FOLDER = None

# If saved per-folder predictions already exist, set this to False to reload instead of recomputing.
FORCE_REEVALUATE = True

# Combined output paths used by notebook 8.
PREDICTIONS_PARQUET = OUTPUT_DIR / "eval_predictions_long.parquet"
PREDICTIONS_CSV = OUTPUT_DIR / "eval_predictions_long.csv"
METRICS_PROTEIN_CSV = OUTPUT_DIR / "eval_metrics_by_protein.csv"
METRICS_TISSUE_CSV = OUTPUT_DIR / "eval_metrics_by_tissue.csv"
MODEL_INVENTORY_CSV = OUTPUT_DIR / "model_inventory.csv"
FEATURE_COVERAGE_CSV = OUTPUT_DIR / "feature_coverage.csv"

print("Validation data directory:", VAL_DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())
print("Model directories:")
for label, folder in MODEL_DIRS.items():
    print(f"- {label:14s}: {folder.resolve()} exists={folder.exists()}")


Validation data directory: C:\Users\hp\Desktop\new_pRNA\val_data
Output directory: C:\Users\hp\Desktop\new_pRNA\results_comparison_all_models
Model directories:
- results_xgb   : C:\Users\hp\Desktop\new_pRNA\results_xgb exists=True
- results_enet  : C:\Users\hp\Desktop\new_pRNA\results_enet exists=True
- res_enet_zbig : C:\Users\hp\Desktop\new_pRNA\res_enet_zbig exists=True
- res_xgb_zbig  : C:\Users\hp\Desktop\new_pRNA\res_xgb_zbig exists=True


## 1. Load external validation datasets

In [4]:
def load_val_data(prefix: str):
    rna_path = VAL_DATA_DIR / f"{prefix}_transcriptomics.joblib"
    prot_path = VAL_DATA_DIR / f"{prefix}_proteomics.joblib"

    if not rna_path.exists() or not prot_path.exists():
        print(f"Skipping {prefix}: missing {rna_path.name} or {prot_path.name}")
        return None

    X = joblib.load(rna_path)
    Y = joblib.load(prot_path)

    X = X.copy()
    Y = Y.copy()
    X.index = X.index.map(str)
    X.columns = X.columns.map(str)
    Y.index = Y.index.map(str)
    Y.columns = Y.columns.map(str)

    # Proteomics should be rows=samples, columns=proteins. If needed, transpose.
    overlap_rows = len(set(X.index).intersection(set(Y.index)))
    overlap_cols = len(set(X.index).intersection(set(Y.columns)))
    if overlap_cols > overlap_rows:
        Y = Y.T.copy()
        Y.index = Y.index.map(str)
        Y.columns = Y.columns.map(str)

    common = X.index.intersection(Y.index)
    X = X.loc[common].copy()
    Y = Y.loc[common].copy()

    print(f"Loaded {prefix:10s}: X={X.shape}, Y={Y.shape}, common samples={len(common)}")
    return X, Y


datasets = {}
for prefix in DATASET_PREFIXES:
    loaded = load_val_data(prefix)
    if loaded is not None:
        datasets[prefix] = loaded

if not datasets:
    raise FileNotFoundError("No validation datasets were loaded. Check VAL_DATA_DIR and DATASET_PREFIXES.")

print("\nDatasets loaded:", list(datasets.keys()))


Loaded LabelFree : X=(360, 11822), Y=(360, 2155), common samples=360
Loaded iTRAQ     : X=(276, 11822), Y=(276, 2155), common samples=276
Loaded TMT       : X=(1924, 11822), Y=(1924, 2155), common samples=1924

Datasets loaded: ['LabelFree', 'iTRAQ', 'TMT']


## 2. Discover model H5 files

In [5]:
def parse_model_filename(path: Path):
    stem = path.stem
    protein = stem
    model_family = "unknown"
    normalization = "unknown"
    grid_preset = "unknown"
    select_k = np.nan

    if "_xgb_8020_" in stem:
        protein = stem.split("_xgb_8020_")[0]
        model_family = "xgb"
        rest = stem.split("_xgb_8020_")[1]
    elif "_enet_8020_" in stem:
        protein = stem.split("_enet_8020_")[0]
        model_family = "enet"
        rest = stem.split("_enet_8020_")[1]
    else:
        rest = stem

    toks = rest.split("_")
    known_norms = {"none", "zscore", "rank", "ranking", "binning", "qbin5", "qbin10", "qbin20", "qbin50", "qbin100"}
    for tok in toks:
        if tok in known_norms or re.fullmatch(r"qbin\d+", tok):
            normalization = tok
            break
    for tok in toks:
        if tok in {"tiny", "fast", "current"}:
            grid_preset = tok
            break
    for tok in toks:
        m = re.fullmatch(r"k(\d+)", tok)
        if m:
            select_k = int(m.group(1))
            break

    return protein, model_family, normalization, grid_preset, select_k


model_files = []
for source_label, folder in MODEL_DIRS.items():
    files = sorted(folder.glob("*.h5")) if folder.exists() else []
    if MAX_MODELS_PER_FOLDER is not None:
        files = files[:int(MAX_MODELS_PER_FOLDER)]
    print(f"{source_label:10s}: {len(files):6d} H5 files")
    for p in files:
        protein, family, norm, grid, k = parse_model_filename(p)
        model_files.append({
            "source_folder": source_label,
            "h5_path": str(p),
            "file": p.name,
            "protein": protein,
            "model_family": family,
            "normalization": norm,
            "grid_preset": grid,
            "select_k": k,
            "file_size_mb": p.stat().st_size / (1024 ** 2),
        })

model_inventory = pd.DataFrame(model_files)
if MAX_MODELS_TOTAL is not None:
    model_inventory = model_inventory.head(int(MAX_MODELS_TOTAL)).copy()

if model_inventory.empty:
    raise FileNotFoundError("No model H5 files found in MODEL_DIRS.")

model_inventory["run_label"] = model_inventory.apply(
    lambda r: f"{r['source_folder']} | {r['normalization']} | {r['grid_preset']} | k{int(r['select_k']) if pd.notna(r['select_k']) else 'NA'}",
    axis=1,
)

model_inventory.to_csv(MODEL_INVENTORY_CSV, index=False)

print("\nTotal models selected for evaluation:", len(model_inventory))
print("Unique proteins:", model_inventory["protein"].nunique())
print("Run labels:")
display(model_inventory.groupby("run_label").agg(models=("file", "count"), proteins=("protein", "nunique")).sort_values("models", ascending=False))


results_xgb:  20462 H5 files
results_enet:  20602 H5 files
res_enet_zbig:   5113 H5 files
res_xgb_zbig:   5113 H5 files

Total models selected for evaluation: 51290
Unique proteins: 5113
Run labels:


,models,proteins
run_label,,
res_enet_zbig | zscore | fast | k5000,5113,5113
res_xgb_zbig | zscore | current | k5000,5113,5113
results_enet | none | fast | k2000,5113,5113
results_enet | ranking | fast | k2000,5113,5113
results_enet | unknown | fast | k2000,5113,5113
results_enet | zscore | fast | k2000,5113,5113
results_xgb | none | fast | k2000,5113,5113
results_xgb | qbin50 | fast | k2000,5113,5113
results_xgb | rank | fast | k2000,5113,5113


## 3. Prediction helpers

In [6]:
def load_artifact_from_h5(path):
    with h5py.File(path, "r") as h5:
        payload = h5["model/artifact_pickle_xz"][:]
    return pickle.loads(lzma.decompress(np.asarray(payload, dtype=np.uint8).tobytes()))


def _as_finite_array(values, default=0.0):
    arr = np.asarray(values, dtype=np.float32)
    arr[~np.isfinite(arr)] = default
    return arr


def apply_row_binning(X_full: pd.DataFrame, n_bins: int = 50) -> pd.DataFrame:
    """Row-wise relative representation binning. Used for ENet binning when no fitted edges are stored."""
    x = X_full.to_numpy(dtype=np.float64, copy=True)
    x[x == 0] = np.nan
    qs = np.linspace(0.0, 1.0, int(n_bins), dtype=np.float64)
    q = np.nanquantile(x, qs, axis=1).T
    out = np.zeros_like(x, dtype=np.float32)
    for i in range(x.shape[0]):
        row = x[i]
        m = np.isfinite(row)
        if not m.any() or np.isnan(q[i]).all():
            continue
        idx = np.searchsorted(q[i], row[m], side="left")
        out[i, m] = float(n_bins) - idx.astype(np.float32)
    return pd.DataFrame(out, index=X_full.index, columns=X_full.columns)


def apply_row_ranking(X_full: pd.DataFrame) -> pd.DataFrame:
    x = X_full.to_numpy(dtype=np.float64, copy=False)
    ranked = x.argsort(axis=1).argsort(axis=1).astype(np.float32, copy=False)
    return pd.DataFrame(ranked, index=X_full.index, columns=X_full.columns)


def apply_artifact_normalizer(X_new: pd.DataFrame, artifact: dict) -> tuple:
    """Return transformed design matrix in the exact selected-feature order plus coverage stats."""
    selected = list(map(str, artifact.get("selected_features", [])))
    if not selected:
        raise ValueError("Artifact has no selected_features.")

    normalization = str(artifact.get("normalization", "none")).lower()
    normalizer = artifact.get("normalizer", None) or {}
    method = str(normalizer.get("method", normalization)).lower() if isinstance(normalizer, dict) else normalization

    missing_features = [g for g in selected if g not in X_new.columns]
    missing_fraction = len(missing_features) / max(1, len(selected))

    # XGB qbin/rank/zscore style normalizers.
    if isinstance(normalizer, dict) and method == "zscore" and "mean" in normalizer and "std" in normalizer:
        cols = list(map(str, normalizer.get("columns", selected)))
        Xraw = X_new.reindex(columns=cols)
        mean = np.asarray(normalizer["mean"], dtype=np.float32)
        std = np.asarray(normalizer["std"], dtype=np.float32)
        std[~np.isfinite(std) | (std == 0)] = 1.0
        arr = Xraw.to_numpy(dtype=np.float32, copy=True)
        # fill missing values with train mean so transformed value is 0
        for j in range(arr.shape[1]):
            bad = ~np.isfinite(arr[:, j])
            if bad.any():
                arr[bad, j] = mean[j] if j < len(mean) else 0.0
        out = ((arr - mean) / std).astype(np.float32)
        return out, {"selected_feature_count": len(cols), "missing_feature_count": len(missing_features), "missing_feature_fraction": missing_fraction}

    if isinstance(normalizer, dict) and method in {"rank", "ranking"} and "sorted_values" in normalizer:
        cols = list(map(str, normalizer.get("columns", selected)))
        Xraw = X_new.reindex(columns=cols)
        out = np.zeros((Xraw.shape[0], len(cols)), dtype=np.float32)
        for j, col in enumerate(cols):
            sorted_col = np.asarray(normalizer.get("sorted_values", {}).get(col, []), dtype=np.float32)
            vals = Xraw.iloc[:, j].to_numpy(dtype=np.float32, copy=False)
            mask = np.isfinite(vals)
            if sorted_col.size > 0:
                out[mask, j] = np.searchsorted(sorted_col, vals[mask], side="right") / float(sorted_col.size)
        return out, {"selected_feature_count": len(cols), "missing_feature_count": len(missing_features), "missing_feature_fraction": missing_fraction}

    if isinstance(normalizer, dict) and (method.startswith("qbin") or "edges" in normalizer):
        cols = list(map(str, normalizer.get("columns", selected)))
        Xraw = X_new.reindex(columns=cols)
        n_bins = int(normalizer.get("n_bins", re.sub("[^0-9]", "", method) or 50))
        denom = max(1, n_bins - 1)
        out = np.zeros((Xraw.shape[0], len(cols)), dtype=np.float32)
        for j, col in enumerate(cols):
            edges = np.asarray(normalizer.get("edges", {}).get(col, []), dtype=np.float32)
            vals = Xraw.iloc[:, j].to_numpy(dtype=np.float32, copy=False)
            mask = np.isfinite(vals)
            if edges.size > 0:
                out[mask, j] = np.searchsorted(edges, vals[mask], side="right") / float(denom)
        return out, {"selected_feature_count": len(cols), "missing_feature_count": len(missing_features), "missing_feature_fraction": missing_fraction}

    # ENet row-wise relative representations.
    if normalization in {"binning", "relative_binning"} or method in {"binning", "relative_binning"}:
        n_bins = int(artifact.get("n_bins", normalizer.get("n_bins", 50) if isinstance(normalizer, dict) else 50))
        Xtrans = apply_row_binning(X_new, n_bins=n_bins).reindex(columns=selected, fill_value=0.0)
        return Xtrans.to_numpy(dtype=np.float32, copy=False), {"selected_feature_count": len(selected), "missing_feature_count": len(missing_features), "missing_feature_fraction": missing_fraction}

    if normalization in {"ranking", "rank"} or method in {"ranking", "rank"}:
        Xtrans = apply_row_ranking(X_new).reindex(columns=selected, fill_value=0.0)
        return Xtrans.to_numpy(dtype=np.float32, copy=False), {"selected_feature_count": len(selected), "missing_feature_count": len(missing_features), "missing_feature_fraction": missing_fraction}

    # No normalizer or simple none. Missing genes become zeros.
    Xmat = X_new.reindex(columns=selected, fill_value=0.0)
    arr = Xmat.to_numpy(dtype=np.float32, copy=True)
    arr[~np.isfinite(arr)] = 0.0
    return arr, {"selected_feature_count": len(selected), "missing_feature_count": len(missing_features), "missing_feature_fraction": missing_fraction}


def evaluate_metric_vector(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    out = {
        "mse": np.nan,
        "pearson": np.nan,
        "cosine_similarity": np.nan,
        "n_used": int(mask.sum()),
    }
    if mask.sum() == 0:
        return out

    yt = y_true[mask]
    yp = y_pred[mask]
    out["mse"] = float(np.mean((yt - yp) ** 2))
    if len(yt) >= 2 and np.std(yt) > 0 and np.std(yp) > 0:
        out["pearson"] = float(np.corrcoef(yt, yp)[0, 1])
    denom = float(np.linalg.norm(yt) * np.linalg.norm(yp))
    if denom > 0:
        out["cosine_similarity"] = float(np.dot(yt, yp) / denom)
    return out


def infer_tissue(sample_id: str) -> str:
    """Simple tissue inference from sample IDs. Customize this if you have a better tissue map."""
    s = str(sample_id)
    if "_" in s:
        tail = s.split("_")[-1]
        if 2 <= len(tail) <= 30:
            return tail
    if "-" in s:
        return s.split("-")[0]
    return "Unknown"


## 4. Split evaluation helpers

The helper below evaluates one model folder at a time, saves that folder's results immediately, and then rebuilds the combined files from any completed folders.


In [9]:
PREDICTION_COLUMNS = [
    "dataset", "sample", "tissue", "protein", "y_true", "y_pred", "source_folder",
    "model_family", "normalization", "grid_preset", "select_k", "run_label", "h5_path",
]

METRIC_COLUMNS = [
    "dataset", "protein", "source_folder", "model_family", "normalization", "grid_preset",
    "select_k", "run_label", "h5_path", "test_mse", "test_pearson_corr",
    "test_cosine_similarity", "n_test_used", "selected_feature_count",
    "missing_feature_count", "missing_feature_fraction",
]

COVERAGE_COLUMNS = [
    "dataset", "protein", "source_folder", "run_label", "h5_path", "selected_feature_count",
    "missing_feature_count", "missing_feature_fraction",
]


def source_output_paths(source_folder: str) -> dict:
    """Return all per-folder output paths for a model source."""
    stem = str(source_folder)
    return {
        "predictions_parquet": OUTPUT_DIR / f"{stem}__eval_predictions_long.parquet",
        "predictions_csv": OUTPUT_DIR / f"{stem}__eval_predictions_long.csv",
        "metrics_protein_csv": OUTPUT_DIR / f"{stem}__eval_metrics_by_protein.csv",
        "feature_coverage_csv": OUTPUT_DIR / f"{stem}__feature_coverage.csv",
    }


def write_predictions_with_fallback(df: pd.DataFrame, parquet_path: Path, csv_path: Path) -> Path:
    """Save predictions as parquet when possible, otherwise CSV."""
    if df is None:
        df = pd.DataFrame()
    try:
        df.to_parquet(parquet_path, index=False)
        if csv_path.exists():
            # Avoid confusion from stale CSV fallbacks when parquet succeeds.
            csv_path.unlink()
        print("Saved:", parquet_path)
        return parquet_path
    except Exception as e:
        df.to_csv(csv_path, index=False)
        print("Parquet failed; saved CSV:", csv_path, "error:", e)
        return csv_path


def read_saved_predictions(parquet_path: Path, csv_path: Path) -> pd.DataFrame:
    """Load saved predictions from parquet if available, with CSV fallback."""
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f"Could not read {parquet_path.name}; trying CSV fallback. Error: {e}")
    if csv_path.exists():
        try:
            return pd.read_csv(csv_path)
        except pd.errors.EmptyDataError:
            return pd.DataFrame(columns=PREDICTION_COLUMNS)
    return pd.DataFrame(columns=PREDICTION_COLUMNS)


def read_csv_if_exists(path: Path, columns: list) -> pd.DataFrame:
    """Read a CSV if it exists; return an empty dataframe with columns otherwise."""
    if not path.exists():
        return pd.DataFrame(columns=columns)
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=columns)


def evaluate_model_inventory_subset(sub_inventory: pd.DataFrame, desc: str):
    """Evaluate a subset of model_inventory and return predictions, protein metrics, and coverage."""
    prediction_chunks = []
    protein_metric_rows = []
    coverage_rows = []

    sub_inventory = sub_inventory.copy().reset_index(drop=True)
    if sub_inventory.empty:
        print(f"No models to evaluate for: {desc}")
        return pd.DataFrame(columns=PREDICTION_COLUMNS), pd.DataFrame(columns=METRIC_COLUMNS), pd.DataFrame(columns=COVERAGE_COLUMNS)

    iterator = sub_inventory.itertuples(index=False)
    for row in tqdm(list(iterator), total=len(sub_inventory), desc=desc):
        h5_path = Path(row.h5_path)
        try:
            artifact = load_artifact_from_h5(h5_path)
            selected = list(map(str, artifact.get("selected_features", [])))
            if not selected:
                raise ValueError("No selected_features in artifact")
        except Exception as e:
            print(f"Skipping {row.file}: could not load artifact: {type(e).__name__}: {e}")
            continue

        protein = str(getattr(row, "protein"))
        model = artifact.get("model", None)
        if model is None:
            print(f"Skipping {row.file}: artifact has no model")
            continue

        for dataset_name, (X_ds, Y_ds) in datasets.items():
            if protein not in Y_ds.columns:
                continue
            try:
                Xmat, coverage = apply_artifact_normalizer(X_ds, artifact)
                y_pred = model.predict(Xmat)
            except Exception:
                # Try dataframe fallback for rare sklearn/xgb behavior.
                try:
                    X_df = X_ds.reindex(columns=selected, fill_value=0.0)
                    y_pred = model.predict(X_df)
                    coverage = {
                        "selected_feature_count": len(selected),
                        "missing_feature_count": len([g for g in selected if g not in X_ds.columns]),
                        "missing_feature_fraction": np.nan,
                    }
                except Exception as e:
                    print(f"Skipping {row.file} on {dataset_name}: prediction failed: {type(e).__name__}: {e}")
                    continue

            y_true = Y_ds[protein].to_numpy(dtype=float)
            sample_ids = Y_ds.index.astype(str).to_numpy()
            tissues = [infer_tissue(s) for s in sample_ids]

            pred_df = pd.DataFrame({
                "dataset": dataset_name,
                "sample": sample_ids,
                "tissue": tissues,
                "protein": protein,
                "y_true": y_true,
                "y_pred": np.asarray(y_pred, dtype=float),
                "source_folder": row.source_folder,
                "model_family": row.model_family,
                "normalization": row.normalization,
                "grid_preset": row.grid_preset,
                "select_k": row.select_k,
                "run_label": row.run_label,
                "h5_path": str(h5_path),
            })
            prediction_chunks.append(pred_df)

            m = evaluate_metric_vector(y_true, y_pred)
            metric_row = {
                "dataset": dataset_name,
                "protein": protein,
                "source_folder": row.source_folder,
                "model_family": row.model_family,
                "normalization": row.normalization,
                "grid_preset": row.grid_preset,
                "select_k": row.select_k,
                "run_label": row.run_label,
                "h5_path": str(h5_path),
                "test_mse": m["mse"],
                "test_pearson_corr": m["pearson"],
                "test_cosine_similarity": m["cosine_similarity"],
                "n_test_used": m["n_used"],
                **coverage,
            }
            protein_metric_rows.append(metric_row)
            coverage_rows.append({
                "dataset": dataset_name,
                "protein": protein,
                "source_folder": row.source_folder,
                "run_label": row.run_label,
                "h5_path": str(h5_path),
                **coverage,
            })

    predictions = pd.concat(prediction_chunks, ignore_index=True) if prediction_chunks else pd.DataFrame(columns=PREDICTION_COLUMNS)
    metrics = pd.DataFrame(protein_metric_rows, columns=METRIC_COLUMNS)
    coverage = pd.DataFrame(coverage_rows, columns=COVERAGE_COLUMNS)
    return predictions, metrics, coverage


def evaluate_and_save_source(source_folder: str, force: bool = None) -> dict:
    """Evaluate one folder, save its results immediately, and return a result bundle."""
    if force is None:
        force = FORCE_REEVALUATE
    if source_folder not in MODEL_DIRS:
        raise KeyError(f"Unknown source_folder {source_folder!r}. Expected one of: {list(MODEL_DIRS)}")

    paths = source_output_paths(source_folder)
    can_reload = (
        (paths["predictions_parquet"].exists() or paths["predictions_csv"].exists())
        and paths["metrics_protein_csv"].exists()
        and paths["feature_coverage_csv"].exists()
    )

    if (not force) and can_reload:
        print(f"Loading existing saved results for {source_folder}")
        predictions = read_saved_predictions(paths["predictions_parquet"], paths["predictions_csv"])
        metrics = read_csv_if_exists(paths["metrics_protein_csv"], METRIC_COLUMNS)
        coverage = read_csv_if_exists(paths["feature_coverage_csv"], COVERAGE_COLUMNS)
    else:
        sub_inventory = model_inventory[model_inventory["source_folder"].eq(source_folder)].copy()
        print(f"\nEvaluating {source_folder}: {len(sub_inventory)} model files")
        predictions, metrics, coverage = evaluate_model_inventory_subset(
            sub_inventory,
            desc=f"Evaluating {source_folder}",
        )

        print(f"\nSaving per-folder results for {source_folder}...")
        write_predictions_with_fallback(predictions, paths["predictions_parquet"], paths["predictions_csv"])
        metrics.to_csv(paths["metrics_protein_csv"], index=False)
        coverage.to_csv(paths["feature_coverage_csv"], index=False)
        print("Saved:", paths["metrics_protein_csv"])
        print("Saved:", paths["feature_coverage_csv"])

    print(f"\nFinished {source_folder}")
    print("Prediction rows:", len(predictions))
    print("Protein metric rows:", len(metrics))
    if len(metrics) and "dataset" in metrics:
        print("Datasets:", sorted(metrics["dataset"].dropna().unique()))

    return {
        "source_folder": source_folder,
        "predictions": predictions,
        "metrics_by_protein": metrics,
        "feature_coverage": coverage,
        "paths": paths,
    }


def refresh_combined_results(source_folders=None):
    """Rebuild combined notebook-8 files from all per-folder results that currently exist."""
    if source_folders is None:
        source_folders = EVALUATION_ORDER

    prediction_parts = []
    metric_parts = []
    coverage_parts = []
    loaded_sources = []

    for source_folder in source_folders:
        paths = source_output_paths(source_folder)
        preds = read_saved_predictions(paths["predictions_parquet"], paths["predictions_csv"])
        metrics_path = paths["metrics_protein_csv"]
        coverage_path = paths["feature_coverage_csv"]

        if preds.empty and not metrics_path.exists():
            continue

        if not preds.empty:
            prediction_parts.append(preds)
        if metrics_path.exists():
            metric_parts.append(read_csv_if_exists(metrics_path, METRIC_COLUMNS))
        if coverage_path.exists():
            coverage_parts.append(read_csv_if_exists(coverage_path, COVERAGE_COLUMNS))
        loaded_sources.append(source_folder)

    predictions = pd.concat(prediction_parts, ignore_index=True) if prediction_parts else pd.DataFrame(columns=PREDICTION_COLUMNS)
    metrics = pd.concat(metric_parts, ignore_index=True) if metric_parts else pd.DataFrame(columns=METRIC_COLUMNS)
    coverage = pd.concat(coverage_parts, ignore_index=True) if coverage_parts else pd.DataFrame(columns=COVERAGE_COLUMNS)

    print("\nRefreshing combined outputs from completed sources:", loaded_sources)
    write_predictions_with_fallback(predictions, PREDICTIONS_PARQUET, PREDICTIONS_CSV)
    metrics.to_csv(METRICS_PROTEIN_CSV, index=False)
    coverage.to_csv(FEATURE_COVERAGE_CSV, index=False)
    print("Saved:", METRICS_PROTEIN_CSV)
    print("Saved:", FEATURE_COVERAGE_CSV)

    return predictions, metrics, coverage


<div style="padding:16px; border-left:9px solid #1f77b4; background:#f7fbff; border-radius:10px; margin:10px 0;">

## 🟦 Evaluation: `results_xgb`

Runs only models from `<code>../results_xgb</code>` and saves this folder's predictions, protein metrics, and feature coverage as soon as the cell finishes. The combined notebook-8 files are refreshed from all completed sections afterward.

</div>


In [11]:
# ============================================================
# Evaluation 1/4: results_xgb
# ============================================================

results_xgb = evaluate_and_save_source("results_xgb")
predictions_long, metrics_by_protein, feature_coverage = refresh_combined_results()

print("\nCombined prediction rows:", len(predictions_long))
print("Combined protein metric rows:", len(metrics_by_protein))
display(results_xgb["metrics_by_protein"].head())



Evaluating results_xgb: 20462 model files


Evaluating results_xgb:   0%|          | 0/20462 [00:00<?, ?it/s]


Saving per-folder results for results_xgb...
Saved: ..\results_comparison_all_models\results_xgb__eval_predictions_long.parquet
Saved: ..\results_comparison_all_models\results_xgb__eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\results_xgb__feature_coverage.csv

Finished results_xgb
Prediction rows: 22072320
Protein metric rows: 25866
Datasets: ['LabelFree', 'TMT', 'iTRAQ']

Refreshing combined outputs from completed sources: ['results_xgb', 'results_enet', 'res_enet_zbig', 'res_xgb_zbig']
Saved: ..\results_comparison_all_models\eval_predictions_long.parquet
Saved: ..\results_comparison_all_models\eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\feature_coverage.csv

Combined prediction rows: 33105920
Combined protein metric rows: 64881


,dataset,protein,source_folder,model_family,normalization,grid_preset,select_k,run_label,h5_path,test_mse,test_pearson_corr,test_cosine_similarity,n_test_used,selected_feature_count,missing_feature_count,missing_feature_fraction
0,LabelFree,A2M,results_xgb,xgb,none,fast,2000,results_xgb | none | fast | k2000,..\results_xgb\A2M_xgb_8020_none_fast_k2000.h5,0.309270,0.216253,0.371375,360,2000,668,0.334
1,iTRAQ,A2M,results_xgb,xgb,none,fast,2000,results_xgb | none | fast | k2000,..\results_xgb\A2M_xgb_8020_none_fast_k2000.h5,0.688956,0.086966,0.010461,276,2000,668,0.334
2,TMT,A2M,results_xgb,xgb,none,fast,2000,results_xgb | none | fast | k2000,..\results_xgb\A2M_xgb_8020_none_fast_k2000.h5,5.736022,-0.368703,-0.043190,1924,2000,668,0.334
3,LabelFree,A2M,results_xgb,xgb,none,tiny,50,results_xgb | none | tiny | k50,..\results_xgb\A2M_xgb_8020_none_tiny_k50.h5,0.292603,0.067915,-0.132168,360,50,6,0.120
4,iTRAQ,A2M,results_xgb,xgb,none,tiny,50,results_xgb | none | tiny | k50,..\results_xgb\A2M_xgb_8020_none_tiny_k50.h5,0.518164,0.229213,0.056677,276,50,6,0.120


<div style="padding:16px; border-left:9px solid #2ca02c; background:#f7fbff; border-radius:10px; margin:10px 0;">

## 🟩 Evaluation: `results_enet`

Runs only models from `<code>../results_enet</code>` and saves this folder's predictions, protein metrics, and feature coverage as soon as the cell finishes. The combined notebook-8 files are refreshed from all completed sections afterward.

</div>


In [10]:
# ============================================================
# Evaluation 2/4: results_enet
# ============================================================

Results_enet = evaluate_and_save_source("results_enet")
results_enet = Results_enet
predictions_long, metrics_by_protein, feature_coverage = refresh_combined_results()

print("\nCombined prediction rows:", len(predictions_long))
print("Combined protein metric rows:", len(metrics_by_protein))
display(Results_enet["metrics_by_protein"].head())



Evaluating results_enet: 20602 model files


Evaluating results_enet:   0%|          | 0/20602 [00:00<?, ?it/s]


Saving per-folder results for results_enet...
Parquet failed; saved CSV: ..\results_comparison_all_models\results_enet__eval_predictions_long.csv error: malloc of size 2147483648 failed
Saved: ..\results_comparison_all_models\results_enet__eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\results_enet__feature_coverage.csv

Finished results_enet
Prediction rows: 22259200
Protein metric rows: 26085
Datasets: ['LabelFree', 'TMT', 'iTRAQ']

Refreshing combined outputs from completed sources: ['results_enet', 'res_enet_zbig', 'res_xgb_zbig']
Saved: ..\results_comparison_all_models\eval_predictions_long.parquet
Saved: ..\results_comparison_all_models\eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\feature_coverage.csv

Combined prediction rows: 11033600
Combined protein metric rows: 39015


,dataset,protein,source_folder,model_family,normalization,grid_preset,select_k,run_label,h5_path,test_mse,test_pearson_corr,test_cosine_similarity,n_test_used,selected_feature_count,missing_feature_count,missing_feature_fraction
0,LabelFree,A2M,results_enet,enet,unknown,fast,2000,results_enet | unknown | fast | k2000,..\results_enet\A2M_enet_8020_binning10_fast_k...,9.543728,-0.234928,-0.406378,360,2000,506,0.253
1,iTRAQ,A2M,results_enet,enet,unknown,fast,2000,results_enet | unknown | fast | k2000,..\results_enet\A2M_enet_8020_binning10_fast_k...,2.127356,-0.139304,-0.129947,276,2000,506,0.253
2,TMT,A2M,results_enet,enet,unknown,fast,2000,results_enet | unknown | fast | k2000,..\results_enet\A2M_enet_8020_binning10_fast_k...,15.157238,-0.067372,-0.030630,1924,2000,506,0.253
3,LabelFree,A2M,results_enet,enet,unknown,fast,500,results_enet | unknown | fast | k500,..\results_enet\A2M_enet_8020_binning10_fast_k...,7.699465,-0.131402,-0.296666,360,500,82,0.164
4,iTRAQ,A2M,results_enet,enet,unknown,fast,500,results_enet | unknown | fast | k500,..\results_enet\A2M_enet_8020_binning10_fast_k...,4.394886,-0.080707,-0.063859,276,500,82,0.164


<div style="padding:16px; border-left:9px solid #ffbf00; background:#f7fbff; border-radius:10px; margin:10px 0;">

## 🟨 Evaluation: `res_enet_zbig`

Runs only models from `<code>../res_enet_zbig</code>` and saves this folder's predictions, protein metrics, and feature coverage as soon as the cell finishes. The combined notebook-8 files are refreshed from all completed sections afterward.

</div>


In [8]:
# ============================================================
# Evaluation 3/4: res_enet_zbig
# ============================================================

res_enet_zbig = evaluate_and_save_source("res_enet_zbig")
predictions_long, metrics_by_protein, feature_coverage = refresh_combined_results()

print("\nCombined prediction rows:", len(predictions_long))
print("Combined protein metric rows:", len(metrics_by_protein))
display(res_enet_zbig["metrics_by_protein"].head())



Evaluating res_enet_zbig: 5113 model files


Evaluating res_enet_zbig:   0%|          | 0/5113 [00:00<?, ?it/s]


Saving per-folder results for res_enet_zbig...
Saved: ..\results_comparison_all_models\res_enet_zbig__eval_predictions_long.parquet
Saved: ..\results_comparison_all_models\res_enet_zbig__eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\res_enet_zbig__feature_coverage.csv

Finished res_enet_zbig
Prediction rows: 5516800
Protein metric rows: 6465
Datasets: ['LabelFree', 'TMT', 'iTRAQ']

Refreshing combined outputs from completed sources: ['res_enet_zbig']
Saved: ..\results_comparison_all_models\eval_predictions_long.parquet
Saved: ..\results_comparison_all_models\eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\feature_coverage.csv

Combined prediction rows: 5516800
Combined protein metric rows: 6465


,dataset,protein,source_folder,model_family,normalization,grid_preset,select_k,run_label,h5_path,test_mse,test_pearson_corr,test_cosine_similarity,n_test_used,selected_feature_count,missing_feature_count,missing_feature_fraction
0,LabelFree,A2M,res_enet_zbig,enet,zscore,fast,5000,res_enet_zbig | zscore | fast | k5000,..\res_enet_zbig\A2M_enet_8020_zscore_fast_k50...,4.436586,-0.204794,-0.184042,360,5000,1820,NaN
1,iTRAQ,A2M,res_enet_zbig,enet,zscore,fast,5000,res_enet_zbig | zscore | fast | k5000,..\res_enet_zbig\A2M_enet_8020_zscore_fast_k50...,3.138782,-0.088701,-0.088663,276,5000,1820,NaN
2,TMT,A2M,res_enet_zbig,enet,zscore,fast,5000,res_enet_zbig | zscore | fast | k5000,..\res_enet_zbig\A2M_enet_8020_zscore_fast_k50...,7.063206,0.230085,0.230022,1924,5000,1820,NaN
3,LabelFree,AARS2,res_enet_zbig,enet,zscore,fast,5000,res_enet_zbig | zscore | fast | k5000,..\res_enet_zbig\AARS2_enet_8020_zscore_fast_k...,1.369630,0.427378,0.408251,360,5000,2115,NaN
4,iTRAQ,AARS2,res_enet_zbig,enet,zscore,fast,5000,res_enet_zbig | zscore | fast | k5000,..\res_enet_zbig\AARS2_enet_8020_zscore_fast_k...,0.310668,0.028896,0.017951,276,5000,2115,NaN


<div style="padding:16px; border-left:9px solid #ff7f0e; background:#f7fbff; border-radius:10px; margin:10px 0;">

## 🟧 Evaluation: `res_xgb_zbig`

Runs only models from `<code>../res_xgb_zbig</code>` and saves this folder's predictions, protein metrics, and feature coverage as soon as the cell finishes. The combined notebook-8 files are refreshed from all completed sections afterward.

</div>


In [9]:
# ============================================================
# Evaluation 4/4: res_xgb_zbig
# ============================================================

res_xgb_zbig = evaluate_and_save_source("res_xgb_zbig")
predictions_long, metrics_by_protein, feature_coverage = refresh_combined_results()

print("\nCombined prediction rows:", len(predictions_long))
print("Combined protein metric rows:", len(metrics_by_protein))
display(res_xgb_zbig["metrics_by_protein"].head())



Evaluating res_xgb_zbig: 5113 model files


Evaluating res_xgb_zbig:   0%|          | 0/5113 [00:00<?, ?it/s]


Saving per-folder results for res_xgb_zbig...
Saved: ..\results_comparison_all_models\res_xgb_zbig__eval_predictions_long.parquet
Saved: ..\results_comparison_all_models\res_xgb_zbig__eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\res_xgb_zbig__feature_coverage.csv

Finished res_xgb_zbig
Prediction rows: 5516800
Protein metric rows: 6465
Datasets: ['LabelFree', 'TMT', 'iTRAQ']

Refreshing combined outputs from completed sources: ['res_enet_zbig', 'res_xgb_zbig']
Saved: ..\results_comparison_all_models\eval_predictions_long.parquet
Saved: ..\results_comparison_all_models\eval_metrics_by_protein.csv
Saved: ..\results_comparison_all_models\feature_coverage.csv

Combined prediction rows: 11033600
Combined protein metric rows: 12930


,dataset,protein,source_folder,model_family,normalization,grid_preset,select_k,run_label,h5_path,test_mse,test_pearson_corr,test_cosine_similarity,n_test_used,selected_feature_count,missing_feature_count,missing_feature_fraction
0,LabelFree,A2M,res_xgb_zbig,xgb,zscore,current,5000,res_xgb_zbig | zscore | current | k5000,..\res_xgb_zbig\A2M_xgb_8020_zscore_current_k5...,0.436974,0.245077,0.414066,360,5000,1938,NaN
1,iTRAQ,A2M,res_xgb_zbig,xgb,zscore,current,5000,res_xgb_zbig | zscore | current | k5000,..\res_xgb_zbig\A2M_xgb_8020_zscore_current_k5...,0.524903,-0.136450,-0.103871,276,5000,1938,NaN
2,TMT,A2M,res_xgb_zbig,xgb,zscore,current,5000,res_xgb_zbig | zscore | current | k5000,..\res_xgb_zbig\A2M_xgb_8020_zscore_current_k5...,5.358588,0.356054,0.131945,1924,5000,1938,NaN
3,LabelFree,AARS2,res_xgb_zbig,xgb,zscore,current,5000,res_xgb_zbig | zscore | current | k5000,..\res_xgb_zbig\AARS2_xgb_8020_zscore_current_...,1.786639,0.194022,0.034785,360,5000,2173,NaN
4,iTRAQ,AARS2,res_xgb_zbig,xgb,zscore,current,5000,res_xgb_zbig | zscore | current | k5000,..\res_xgb_zbig\AARS2_xgb_8020_zscore_current_...,0.454422,0.002494,0.000135,276,5000,2173,NaN


## 5. Combine completed results and compute performance across tissues

In [7]:
def compute_group_metrics(df, group_cols):
    rows = []
    if df is None or df.empty:
        return pd.DataFrame()
    for key, sub in tqdm(df.groupby(group_cols), desc="Computing grouped metrics"):
        if not isinstance(key, tuple):
            key = (key,)
        m = evaluate_metric_vector(sub["y_true"].to_numpy(), sub["y_pred"].to_numpy())
        row = dict(zip(group_cols, key))
        row.update({
            "test_mse": m["mse"],
            "test_pearson_corr": m["pearson"],
            "test_cosine_similarity": m["cosine_similarity"],
            "n_prediction_pairs": m["n_used"],
            "n_samples": sub["sample"].nunique(),
            "n_proteins": sub["protein"].nunique(),
        })
        rows.append(row)
    return pd.DataFrame(rows)


# Make this cell safe to run after a kernel restart, as long as any split-evaluation files exist.
try:
    predictions_long
    metrics_by_protein
except NameError:
    predictions_long, metrics_by_protein, feature_coverage = refresh_combined_results()

if predictions_long.empty:
    raise ValueError("No predictions are available yet. Run at least one highlighted evaluation cell first.")

metrics_by_tissue = compute_group_metrics(
    predictions_long,
    ["dataset", "tissue", "run_label", "source_folder", "model_family", "normalization"],
)
metrics_by_tissue.to_csv(METRICS_TISSUE_CSV, index=False)
print("Saved:", METRICS_TISSUE_CSV)

print("\nTissue metric rows:", len(metrics_by_tissue))
print("Tissues by dataset:")
display(metrics_by_tissue.groupby("dataset")["tissue"].nunique().to_frame("n_tissues"))
display(metrics_by_tissue.head())


NameError: name 'refresh_combined_results' is not defined

## 6. Clean summary prints

In [ ]:
print("Model rows by dataset and run label")
display(
    metrics_by_protein.groupby(["dataset", "run_label"])
    .agg(
        proteins=("protein", "nunique"),
        pearson_nonmissing=("test_pearson_corr", lambda s: s.notna().sum()),
        median_pearson=("test_pearson_corr", "median"),
        median_cosine=("test_cosine_similarity", "median"),
        median_mse=("test_mse", "median"),
    )
    .reset_index()
    .sort_values(["dataset", "median_pearson"], ascending=[True, False])
)

print("\nFeature coverage summary")
display(
    metrics_by_protein.groupby("run_label")
    .agg(
        median_missing_fraction=("missing_feature_fraction", "median"),
        max_missing_fraction=("missing_feature_fraction", "max"),
        median_selected_features=("selected_feature_count", "median"),
    )
    .sort_values("median_missing_fraction")
)


## 7. Visualizations: protein-level external validation performance

In [ ]:
def compact_label(s, max_len=34):
    s = str(s)
    return s if len(s) <= max_len else s[:max_len-1] + "…"

plot_df = metrics_by_protein.copy()
plot_df["run_label_short"] = plot_df["run_label"].map(compact_label)

metric_specs = [
    ("test_pearson_corr", "Pearson correlation", None),
    ("test_cosine_similarity", "Cosine similarity", None),
    ("test_mse", "MSE", "log"),
]

for dataset_name in sorted(plot_df["dataset"].dropna().unique()):
    sub = plot_df[plot_df["dataset"] == dataset_name].copy()
    if sub.empty:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    fig.suptitle(f"Protein-level external validation metrics — {dataset_name}", fontsize=16, fontweight="bold")

    for ax, (metric, label, scale) in zip(axes, metric_specs):
        data = sub.dropna(subset=[metric]).copy()
        if data.empty:
            ax.set_title(f"{label}\n(no valid values)")
            ax.axis("off")
            continue
        order = data.groupby("run_label_short")[metric].median().sort_values(ascending=(metric == "test_mse")).index.tolist()
        sns.boxplot(data=data, x="run_label_short", y=metric, order=order, ax=ax, color="#8FB8DE", showfliers=False)
        sns.stripplot(data=data.sample(min(len(data), 1200), random_state=42), x="run_label_short", y=metric, order=order, ax=ax, color="black", size=1.5, alpha=0.18)
        ax.set_title(label)
        ax.set_xlabel("")
        ax.set_ylabel(label)
        ax.tick_params(axis="x", rotation=65)
        if scale == "log":
            ax.set_yscale("log")

    plt.show()


## 8. Visualizations: tissue-level performance

In [ ]:
# Show tissue-level performance in compact heatmaps.
# Rows=tissues, columns=run labels, values=metric.

tissue_plot = metrics_by_tissue.copy()
tissue_plot["run_label_short"] = tissue_plot["run_label"].map(compact_label)

for dataset_name in sorted(tissue_plot["dataset"].dropna().unique()):
    sub_ds = tissue_plot[tissue_plot["dataset"] == dataset_name].copy()
    if sub_ds.empty:
        continue

    # Keep tissues with enough prediction pairs for stable visualization.
    tissue_counts = sub_ds.groupby("tissue")["n_prediction_pairs"].max().sort_values(ascending=False)
    keep_tissues = tissue_counts.head(20).index.tolist()
    sub_ds = sub_ds[sub_ds["tissue"].isin(keep_tissues)].copy()

    fig, axes = plt.subplots(1, 3, figsize=(20, max(5, 0.35 * len(keep_tissues))), constrained_layout=True)
    fig.suptitle(f"Tissue-level external validation performance — {dataset_name}", fontsize=16, fontweight="bold")

    heatmap_specs = [
        ("test_pearson_corr", "Pearson", "viridis", True),
        ("test_cosine_similarity", "Cosine", "viridis", True),
        ("test_mse", "MSE", "mako_r", False),
    ]

    for ax, (metric, title, cmap, higher) in zip(axes, heatmap_specs):
        pivot = sub_ds.pivot_table(index="tissue", columns="run_label_short", values=metric, aggfunc="median")
        if metric == "test_mse":
            # Winsorize display range to keep heatmap readable.
            vals = pivot.to_numpy().ravel()
            vals = vals[np.isfinite(vals)]
            vmax = np.nanpercentile(vals, 90) if len(vals) else None
            sns.heatmap(pivot, ax=ax, cmap=cmap, linewidths=0.4, linecolor="white", cbar_kws={"label": title}, vmax=vmax)
        else:
            sns.heatmap(pivot, ax=ax, cmap=cmap, linewidths=0.4, linecolor="white", cbar_kws={"label": title}, vmin=-1, vmax=1)
        ax.set_title(title)
        ax.set_xlabel("")
        ax.set_ylabel("Tissue")

    plt.show()


## 9. Save-note for downstream notebook

In [ ]:
print("Saved files for notebook 8:")
print("-", PREDICTIONS_PARQUET if PREDICTIONS_PARQUET.exists() else PREDICTIONS_CSV)
print("-", METRICS_PROTEIN_CSV)
print("-", METRICS_TISSUE_CSV)
print("-", MODEL_INVENTORY_CSV)
print("-", FEATURE_COVERAGE_CSV)
print("\nPer-folder checkpoint files are also in:", OUTPUT_DIR.resolve())
print("Open 8.prot_model_val.ipynb next. It reads the combined saved predictions/metrics and creates the final partition plots per validation dataset.")
